# Lesson 04 - Tool Use Design Pattern

In this lesson you will learn the **Tool Use** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- Defining function tools with the `@tool` decorator and typed parameters
- Providing tool schemas so the model knows what each tool does
- Controlling tool execution with `approval_mode`
- Returning **structured output** via Pydantic models and `response_format`

The scenario is a **travel booking agent** that can look up destinations, check availability, and retrieve flight information.

## Setup

In [ ]:
# === Updated 2026-06-09: load .env and ensure the Azure CLI is on PATH ===
import os, shutil
from dotenv import load_dotenv

load_dotenv()  # load repo-root .env into os.environ

# AzureCliCredential shells out to `az`; make sure the kernel can find it.
# (No-op once you fully restart VS Code so it inherits the updated PATH.)
if shutil.which("az") is None:
    os.environ["PATH"] += r";C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin"


In [6]:
# Dependencies already installed in the venv; install left disabled.
# Install the latest agent-framework (>= 1.6.0 ships the modern Foundry API) plus Azure deps.
# %pip install agent-framework azure-ai-projects azure-identity -U -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import logging
import os
import asyncio
from typing import Annotated


# Let the kernel find the Azure CLI (az) that AzureCliCredential shells out to.
# VS Code captured PATH before az was installed, so add its folder explicitly.
os.environ["PATH"] += r";C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin"

from dotenv import load_dotenv
from pydantic import BaseModel
from agent_framework import tool, Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

# Logger namespace moved: "agent_framework.azure" -> "agent_framework.foundry"
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

# Load .env from the repo root into os.environ
load_dotenv()

True

In [12]:
# FoundryChatClient replaces the removed AzureAIProjectAgentProvider.
# It connects directly to your deployed model in the Foundry project
# and reads endpoint + deployment name from the .env values.
client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.

In [13]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent with Multiple Tools

Pass all three tools to the agent so the model can invoke whichever ones it needs to answer the user's question.

In [17]:
travel_tools = [get_destinations, check_availability, get_flight_info]

# Agent() replaces the awaited provider.create_agent().
# Construction is synchronous now; the agent wraps the chat client
# and owns the instructions and tool list.
agent = Agent(
    client=client,
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

Here are the available destinations along with their availability:

1. **Barcelona**: Available
2. **Paris**: Available
3. **Berlin**: Sold out
4. **Tokyo**: Available - 1 spot left
5. **Sydney**: Available
6. **New York City**: Available - 3 spots left

If you're interested in any specific destination, let me know!


## Structured Output with Tools

By passing a Pydantic model as `response_format`, the agent is forced to return a well-typed JSON object instead of free-form text. The parsed model is available on `response.value`. This is useful when downstream code needs to consume the result programmatically.

In [23]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = Agent(
    client=client,
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

# response_format is passed at run time in the options dict.
# With a Pydantic model, the parsed instance is returned on response.value.
response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available.",
    options={"response_format": TravelPlan},
)

if response.value:
    travel_plan = response.value  # TravelPlan instance
    for rec in travel_plan.recommendations:
        print(
            f"{rec.destination}: available={rec.available}, "
            f"{rec.flight_details}, est. ${rec.estimated_cost}"
        )
else:
    print("No structured data found in response")

Barcelona: available=True, No direct flights available., est. $0
Paris: available=True, AF 1081, Departs 09:15, Arrives 11:30, $280, est. $280
Berlin: available=True, BA 2042, Departs 08:30, Arrives 11:45, $350, est. $350


## Tool Approval Patterns

The `approval_mode` parameter on `@tool` controls whether tool calls require human approval before executing:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool runs automatically — no user confirmation needed. |
| `"always_require"` | Every call must be approved by the user before it executes. |

Use `"always_require"` for tools that have side-effects (e.g. booking a flight, charging a credit card) so a human stays in the loop.

In [24]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

Tool name: book_flight
Approval mode: always_require


## Summary

In this lesson you learned how to:

1. **Define tools** using the `@tool` decorator with typed parameters and docstrings that serve as the tool schema.
2. **Compose multiple tools** so the agent can call them in sequence to answer complex queries.
3. **Return structured output** by passing a Pydantic model as `response_format` and reading `response.value`.
4. **Control tool approval** with `approval_mode` to keep a human in the loop for sensitive operations.

These patterns form the foundation for building reliable, production-ready agents that can interact with external systems safely.